# Chi-shape (concave hull)

Implementation of the **chi-shape** (a.k.a. *characteristic shape*) algorithm from:

> Duckham, M., Kulik, L., Worboys, M., & Galton, A. (2008). Efficient generation of simple polygons for characterizing the shape of a set of points in the plane. *Pattern Recognition*, 41(10), 3224-3236.

The algorithm builds the Delaunay triangulation of the input points, then repeatedly removes the longest exterior edge -- subject to a regularity constraint that keeps the boundary a single simple polygon -- until no exterior edge exceeds a length threshold derived from a single normalized parameter `chi_factor` in `[0, 1]`:

- `chi_factor = 1.0` -> the convex hull.
- `chi_factor = 0.0` -> a uniquely-defined, minimum-area simple polygon.

The examples below are chosen to mirror the figures and experiments in the original paper as closely as practical. Every numeric parameter is an interactive slider -- drag them to explore.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
from ipywidgets import interact, IntSlider, FloatSlider

from concavehull import concavehull


def plot_hull(ax, pts, hull, title=None, point_size=8):
    ax.scatter(pts[:, 0], pts[:, 1], s=point_size, color="tab:blue", alpha=0.6)
    ax.plot(hull[:, 0], hull[:, 1], color="tab:red", linewidth=1.5)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_title(title)


def letter_points(letter, n_points, font_size=200, seed=None):
    """Sample `n_points` random points inside the filled outline of `letter`.

    Mirrors the semi-random letter-shaped point sets used throughout both
    papers (e.g. Duckham et al. 2008, Fig. 6/9; Zhong & Duckham 2016, Fig. 3),
    minus the minimum-pairwise-distance constraint they use -- plain uniform
    rejection sampling inside the glyph outline is good enough to show the
    algorithms' behaviour.
    """
    rng = np.random.default_rng(seed)
    prop = FontProperties(family="DejaVu Sans", weight="bold")
    path = TextPath((0, 0), letter, size=font_size, prop=prop)
    (xmin, ymin), (xmax, ymax) = path.get_extents().get_points()
    pts = np.empty((0, 2))
    while len(pts) < n_points:
        batch = rng.uniform([xmin, ymin], [xmax, ymax], size=(4 * n_points, 2))
        inside = path.contains_points(batch)
        pts = np.vstack([pts, batch[inside]])
    pts = pts[:n_points]
    pts = pts - pts.mean(axis=0)
    return pts


def normalize_scale(pts, target_width):
    """Rescale `pts` (already centered) so its widest axis spans `target_width`."""
    extent = np.ptp(pts, axis=0).max()
    return pts * (target_width / extent)

## Example 1: Random point cloud

Basic usage: a plain random Gaussian point cloud, its chi-shape, and the time taken.

In [ ]:
@interact(
    n=IntSlider(value=500, min=50, max=3000, step=50, description="n", continuous_update=False),
    chi_factor=FloatSlider(value=0.05, min=0.0, max=1.0, step=0.01, description="chi_factor",
                            continuous_update=False),
)
def show_basic(n, chi_factor):
    rng = np.random.default_rng(0)
    pts = rng.standard_normal((n, 2))

    start = time.time()
    hull = concavehull(pts, chi_factor=chi_factor)
    elapsed = time.time() - start
    print(f"found concave hull of {n} points in {elapsed:0.5f}s")

    fig, ax = plt.subplots(figsize=(5, 5))
    plot_hull(ax, pts, hull, title=f"Concave Hull (n={n}, chi_factor={chi_factor:.2f})")
    plt.show()

## Example 2: Effect of the `chi_factor` parameter

The paper's Fig. 5 shows a "C"-shaped point distribution carved into progressively more concave shapes as the normalized length parameter decreases from `1.0` (convex hull) towards `0.0`. We reproduce that progression here, using `matplotlib`'s text outlines to build the same kind of letter-shaped point set.

In [ ]:
@interact(
    n_points=IntSlider(value=400, min=100, max=1500, step=50, description="n_points",
                        continuous_update=False),
    chi_factor=FloatSlider(value=0.1, min=0.0, max=1.0, step=0.01, description="chi_factor",
                            continuous_update=False),
)
def show_chi_shape(n_points, chi_factor):
    c_shape = letter_points("C", n_points, seed=1)
    hull = concavehull(c_shape, chi_factor=chi_factor)
    fig, ax = plt.subplots(figsize=(5, 5))
    plot_hull(ax, c_shape, hull, title=f'"C"-shaped points, n={n_points}, chi_factor={chi_factor:.2f}')
    plt.show()

## Example 3: Letter shapes

The paper's main accuracy experiment (Fig. 6) fills the hole-free uppercase letters "C", "F", "G", and "S" with semi-random points and measures how well the chi-shape recovers the letter outline. We reproduce the same four letters here as a small gallery, using a `chi_factor` in the 0.05-0.2 range the paper reports as generally good.

In [ ]:
letters = ["C", "F", "G", "S"]


@interact(
    n_points=IntSlider(value=300, min=100, max=1000, step=50, description="n_points",
                        continuous_update=False),
    chi_factor=FloatSlider(value=0.1, min=0.0, max=1.0, step=0.01, description="chi_factor",
                            continuous_update=False),
)
def show_letter_gallery(n_points, chi_factor):
    fig, axes = plt.subplots(2, 2, figsize=(8, 8))
    for letter, ax in zip(letters, axes.ravel()):
        pts = letter_points(letter, n_points, seed=ord(letter))
        hull = concavehull(pts, chi_factor=chi_factor)
        plot_hull(ax, pts, hull, title=f'"{letter}"')
    fig.suptitle(f"Chi-shapes of letter-shaped point sets (n={n_points}, chi_factor={chi_factor:.2f}), "
                 "cf. Duckham et al. 2008, Fig. 6")
    plt.tight_layout()
    plt.show()

## Example 4: Robustness to point density

Section 6.2 / Fig. 8-9 of the paper investigate how the algorithm behaves as point density changes, finding the optimal parameter to be "relatively stable ... across all point densities". Drag the `n_points` slider at a fixed `chi_factor` to see that same stability yourself.

In [ ]:
@interact(
    n_points=IntSlider(value=500, min=50, max=3000, step=50, description="n_points",
                        continuous_update=False),
    chi_factor=FloatSlider(value=0.1, min=0.0, max=1.0, step=0.01, description="chi_factor",
                            continuous_update=False),
)
def show_density_effect(n_points, chi_factor):
    pts = letter_points("F", n_points, seed=n_points)
    hull = concavehull(pts, chi_factor=chi_factor)
    fig, ax = plt.subplots(figsize=(5, 5))
    plot_hull(ax, pts, hull, title=f'Letter "F", n={n_points}, chi_factor={chi_factor:.2f}')
    plt.show()

## Example 5: Runtime vs. input size

The algorithm is advertised as `O(n log n)`. A quick empirical sanity check, sweeping input size up to `max_size` points:

In [ ]:
@interact(
    chi_factor=FloatSlider(value=0.1, min=0.0, max=1.0, step=0.01, description="chi_factor",
                            continuous_update=False),
    max_size=IntSlider(value=50_000, min=1_000, max=100_000, step=1_000, description="max_size",
                        continuous_update=False),
)
def show_runtime(chi_factor, max_size):
    sizes = sorted(set(np.linspace(100, max_size, 5).astype(int)))
    times = []
    for n in sizes:
        pts = np.random.default_rng(int(n)).standard_normal((n, 2))
        start = time.time()
        concavehull(pts, chi_factor=chi_factor)
        times.append(time.time() - start)
        print(f"n={n:>6}: {times[-1]:0.4f}s")

    fig, ax = plt.subplots()
    ax.plot(sizes, times, marker="o")
    ax.set_xlabel("number of points (n)")
    ax.set_ylabel("time (s)")
    ax.set_title(f"Chi-shape runtime vs. input size (chi_factor={chi_factor:.2f}, expected O(n log n))")
    plt.show()

## Limitation: disconnected clusters and outliers

Chi-shape always returns a *single simple polygon* containing every input point. It has no notion of "outlier" and cannot represent two separate clusters as two separate shapes -- at best it bridges them with a thin connecting strip.

See **`chi_outline.ipynb`** for the Zhong & Duckham (2016) extension that lifts exactly this limitation, and **`bottleneck_split.ipynb`** for splitting a single merged polygon (from either algorithm) into its constituent objects.